In [1]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive/')
os.chdir('/content/drive/MyDrive/AIMEcon_2026/aime-con3')

Mounted at /content/drive/


In [ ]:
%pip install -q "trl==1.13.0" "transformers==5.17.0" "peft==0.21.0" accelerate datasets

In [ ]:
import re
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import start

In [ ]:
os.environ["HF_TOKEN"] = userdata.get("B-Llama-3.1-8B-Access")
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN not set.")

path_to_prompts = start.CODEBOOK_DIR / "llm_prompt_codebook.xlsx"
prompts = pd.read_excel(path_to_prompts)

In [ ]:
with open("requirements.txt", "r") as f:
    print(f.read())

In [ ]:
!pip install -r requirements.txt

In [ ]:
train_df = pd.read_excel(start.DATA_DIR / "train.xlsx")
val_df   = pd.read_excel(start.DATA_DIR / "dev.xlsx")
test_df  = pd.read_excel(start.DATA_DIR / "test.xlsx")

In [ ]:
train_df = train_df.dropna(subset=["text", "code_human"]).reset_index(drop=True)
val_df   = val_df.dropna(subset=["text", "code_human"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["text", "code_human"]).reset_index(drop=True)

In [ ]:
def build_user_prompt(case_text):
    return (
        prompts.loc[prompts.id == "Coding1",   "prompt"].item() + " " +
        prompts.loc[prompts.id == "Construct",  "prompt"].item() + " " +
        prompts.loc[prompts.id == "Prompt1",    "prompt"].item() + " " +
        f'\n"""{case_text}"""\n' +
        prompts.loc[prompts.id == "Format2",    "prompt"].item()
    )

def make_prompt_completion(row, tokenizer):
    label_str = "Yes" if row["code_human"] == 1 else "No"
    messages = [
        {"role": "system",    "content": "You are a helpful text classifier."},
        {"role": "user",      "content": build_user_prompt(str(row["text"]))},
        {"role": "assistant", "content": label_str},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

#Qwen2-7B-Instruct Quantized 4bit

In [ ]:
#Balanced Data

In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers #need to restart the runtime
#!pip install -U peft trl bitsandbytes accelerate transformers need to restart the runtime if not it is not going to work

base qwen 2.5 7B baseline .8321

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.eos_token_id

print(model.get_memory_footprint() / 1e9, "GB")

model neds to be defined before running this as we are using the model tokenizer

In [ ]:
# Balance training set
not_prompt_df = train_df[train_df["code_human"] == 0]
prompt_df     = train_df[train_df["code_human"] == 1]
minority_rows = min(len(not_prompt_df), len(prompt_df))

train_balanced = pd.concat([
    not_prompt_df.sample(n=minority_rows, random_state=42),
    prompt_df.sample(n=minority_rows, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Original   : {len(train_df):,}  {train_df['code_human'].value_counts().to_dict()}")
print(f"Balanced   : {len(train_balanced):,}  {train_balanced['code_human'].value_counts().to_dict()}")
print(f"Val        : {len(val_df):,}  {val_df['code_human'].value_counts().to_dict()}")

train_hf = Dataset.from_list([
    make_prompt_completion(row, tokenizer) for _, row in train_balanced.iterrows()
])
val_hf = Dataset.from_list([
    make_prompt_completion(row, tokenizer) for _, row in val_df.iterrows()
])

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

In [ ]:

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

If 8hs of A100 GPU (...not sure i think with i batch size the model is meemorizing maybe)
    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    max_length=2520,


I think these are better 3h

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    gradient_accumulation_steps=1,

In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
sft_config = SFTConfig(
    output_dir="./Sept_Qwen.2-3B-qlora-dialogic",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    warmup_steps=10,

    learning_rate=2e-4,
    max_grad_norm=0.3,
    fp16=False, bf16=True,
    logging_steps=10,
    eval_strategy="steps", eval_steps=20,
    save_strategy="steps", save_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    max_length=520,              #max_seq_length depending on the package update
    optim= "paged_adamw_8bit", # "adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)]
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/3454 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3454 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3454 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3454 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3454 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/3537 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3537 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/3537 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.735633,0.581045,0.587530,31455.000000,0.878560
40,0.655527,0.562758,0.584819,61105.000000,0.881397
60,0.645536,0.553856,0.560840,91877.000000,0.882312
